# Puntos 10 y 11 – Productos alternativos de precipitación: CHIRPS, ERA5-Land y ERA5
**Trabajo 1 de Hidroclimatología · Cuenca del río Cauca hasta Hidroituango**

- **Punto 10:** descargar la precipitación de cada producto para la cuenca, en un periodo común, y documentar la
  fuente, la resolución espacial, la resolución temporal, el periodo disponible y el periodo seleccionado.
- **Punto 11:** construir el mapa de precipitación media anual multianual de cada producto.

**Periodo común: 1981–2010**, el mismo de las normales del IDEAM. CHIRPS empieza justamente en 1981.

### Antes de correr: cuenta de Copernicus para ERA5 y ERA5-Land
1. Cree una cuenta gratuita en el **Climate Data Store** (cds.climate.copernicus.eu).
2. Entre a las páginas de *ERA5 monthly averaged data on single levels* y de *ERA5-Land monthly averaged data* y
   **acepte la licencia** de cada una, en la pestaña de descarga, al final de la página.
3. Su token ya está en `CDS_KEY`, en la celda de parámetros. La primera descarga lo guarda en el archivo `.cdsapirc` de su usuario; después puede poner `CDS_KEY = None` para que no quede escrito en el notebook.

CHIRPS no necesita cuenta: se descarga ya recortado desde la IRI Data Library (Universidad de Columbia).

**Archivos necesarios:** `DEM_cuenca.tif`, `cuenca_cauca.gpkg` y `P_IDEAM_anual.tif` (para comparar).

**Salidas:**
- `P_CHIRPS_anual.tif`, `P_ERA5L_anual.tif`, `P_ERA5_anual.tif`
- `P_<producto>_mensual.tif` (12 bandas)
- `serie_mensual_cuenca_productos.csv`

Todos los rásters quedan en la grilla del DEM y se usan en los puntos 12 a 14.

## 0. Instalación de librerías
Solo la primera vez. Después de instalar, reinicie el kernel.

In [ ]:
%pip install numpy pandas matplotlib geopandas rasterio pyproj shapely xarray netCDF4 cdsapi requests

## 1. Librerías, parámetros y funciones de apoyo

In [ ]:
import os
import re
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

pd.set_option("display.width", 220)

# ----------------------------- PARÁMETROS -----------------------------
CRS = "EPSG:3116"
CUENCA = "cuenca_cauca.gpkg"
RIO = "rio_principal.gpkg"
DEM = "DEM_cuenca.tif"
P_IDEAM = "P_IDEAM_anual.tif"
PERIODO = (1981, 2010)
MARGEN_GRADOS = 0.5            # margen alrededor de la cuenca para la descarga
CDS_KEY = "3c2fe70f-8070-46fd-bdae-00089dacc3bf"   # token personal del CDS; tras la primera descarga puede volver a None
CARPETA = "productos"
ARCHIVOS = {"CHIRPS": f"{CARPETA}/chirps_{PERIODO[0]}_{PERIODO[1]}.nc",
            "ERA5L": f"{CARPETA}/era5land_tp_{PERIODO[0]}_{PERIODO[1]}.nc",
            "ERA5": f"{CARPETA}/era5_tp_{PERIODO[0]}_{PERIODO[1]}.nc"}
# ----------------------------------------------------------------------

MESES = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
PRODUCTOS = ["CHIRPS", "ERA5L", "ERA5"]
NOMBRE = {"CHIRPS": "CHIRPS v2.0", "ERA5L": "ERA5-Land", "ERA5": "ERA5", "IDEAM": "IDEAM (estaciones)"}
os.makedirs(CARPETA, exist_ok=True)

In [ ]:
# ----------------------- Funciones de apoyo SIG -----------------------
import rasterio
from rasterio import features
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pyproj import Transformer
from shapely.geometry import shape
from shapely.ops import unary_union


def leer_raster(ruta):
    # Devuelve (arreglo enmascarado, info). Si el archivo tiene varias bandas: arreglo (bandas, filas, columnas)
    with rasterio.open(ruta) as src:
        arr = src.read(masked=True).astype("float64")
        info = {"transform": src.transform, "crs": src.crs, "shape": (src.height, src.width),
                "profile": src.profile.copy()}
    return (arr[0] if arr.shape[0] == 1 else arr), info


def guardar_raster(ruta, arr, info, nodata=-9999.0, descripciones=None, dtype="float32"):
    a = np.ma.masked_invalid(np.ma.asarray(arr, dtype="float64"))
    if a.ndim == 2:
        a = a[None]
    perfil = info["profile"].copy()
    perfil.update(driver="GTiff", dtype=dtype, nodata=nodata, count=a.shape[0], compress="lzw")
    with rasterio.open(ruta, "w", **perfil) as dst:
        dst.write(a.filled(nodata).astype(dtype))
        for i, d in enumerate(descripciones or [], start=1):
            dst.set_band_description(i, d)


def coords_celdas(info):
    # Coordenadas (x, y) del centro de cada celda, en el SRC del ráster
    T = info["transform"]
    filas, cols = np.indices(info["shape"])
    return T.a * (cols + 0.5) + T.b * (filas + 0.5) + T.c, T.d * (cols + 0.5) + T.e * (filas + 0.5) + T.f


def a_lonlat(x, y, crs):
    lon, lat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(np.asarray(x), np.asarray(y))
    return np.asarray(lon), np.asarray(lat)


def de_lonlat(lon, lat, crs):
    x, y = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(np.asarray(lon), np.asarray(lat))
    return np.asarray(x), np.asarray(y)


def remuestrear_a_grilla(arr, lon, lat, info, metodo="bilinear"):
    # Lleva un campo en grilla regular lon/lat (2D o 3D con el tiempo primero) a la grilla del DEM
    arr = np.asarray(arr, dtype="float64")
    if lat[0] < lat[-1]:                       # filas de norte a sur
        arr, lat = arr[..., ::-1, :], lat[::-1]
    dx, dy = abs(lon[1] - lon[0]), abs(lat[1] - lat[0])
    src_T = from_origin(lon.min() - dx / 2, lat.max() + dy / 2, dx, dy)
    capas = arr if arr.ndim == 3 else arr[None]
    salida = np.full((capas.shape[0],) + info["shape"], np.nan)
    for k in range(capas.shape[0]):
        reproject(source=np.ascontiguousarray(capas[k]), destination=salida[k], src_transform=src_T,
                  src_crs="EPSG:4326", src_nodata=np.nan, dst_transform=info["transform"], dst_crs=info["crs"],
                  dst_nodata=np.nan, resampling=getattr(Resampling, metodo))
    return salida if arr.ndim == 3 else salida[0]


def mascara_geometria(geom, info):
    # True en las celdas cuyo centro cae dentro de la geometría
    return features.geometry_mask([geom], out_shape=info["shape"], transform=info["transform"], invert=True)


def poligonizar(mascara, info):
    geoms = [shape(g) for g, v in features.shapes(mascara.astype("uint8"), mask=mascara,
                                                    transform=info["transform"]) if v == 1]
    return unary_union(geoms)

In [ ]:
def leer_variable_nc(ruta):
    # Lee la variable de precipitación de un NetCDF sin decodificar fechas (se decodifican aparte)
    import xarray as xr
    ds = xr.open_dataset(ruta, decode_times=False)
    nombre = next((v for v in ("tp", "precipitation", "precip", "prcp") if v in ds.data_vars), None)
    nombre = nombre or max(ds.data_vars, key=lambda v: ds[v].ndim)
    da = ds[nombre]
    out = {"nombre": nombre, "datos": da.values.astype("float64"), "dims": list(da.dims),
           "units": str(da.attrs.get("units", "")),
           "coords": {dim: ds[dim].values for dim in da.dims if dim in ds.variables},
           "unidades_coords": {dim: str(ds[dim].attrs.get("units", "")) for dim in da.dims if dim in ds.variables}}
    ds.close()
    return out

In [ ]:
def decorar(ax, titulo, ext, cuenca, rio=None):
    # Límite de la cuenca, río, flecha de norte, barra de escala y SRC
    cuenca.boundary.plot(ax=ax, color="k", lw=0.8)
    if rio is not None:
        rio.plot(ax=ax, color="#08306b", lw=1.0)
    ax.annotate("", xy=(0.93, 0.93), xytext=(0.93, 0.84), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.5))
    ax.text(0.93, 0.945, "N", transform=ax.transAxes, ha="center", fontsize=12, fontweight="bold")
    x0 = ext[0] + 0.06 * (ext[1] - ext[0]); y0 = ext[2] + 0.04 * (ext[3] - ext[2])
    ax.plot([x0, x0 + 50000], [y0, y0], "k-", lw=3)
    ax.text(x0 + 25000, y0 + 0.012 * (ext[3] - ext[2]), "50 km", ha="center", fontsize=8)
    ax.set_title(titulo, fontsize=11)
    ax.text(0.02, 0.005, "MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)", transform=ax.transAxes, fontsize=7,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))
    ax.ticklabel_format(style="plain"); ax.tick_params(labelsize=7)
    ax.locator_params(axis="x", nbins=4)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_aspect("equal")


def extension(info):
    T = info["transform"]; h, w = info["shape"]
    return [T.c, T.c + T.a * w, T.f + T.e * h, T.f]


def mapa(arr, info, titulo, etiqueta, cuenca, rio=None, cmap="viridis", archivo=None, vmin=None, vmax=None, ax=None):
    ext = extension(info)
    propio = ax is None
    if propio:
        fig, ax = plt.subplots(figsize=(6, 9))
    im = ax.imshow(arr, extent=ext, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    if propio:
        plt.colorbar(im, ax=ax, shrink=0.6, label=etiqueta)
    decorar(ax, titulo, ext, cuenca, rio)
    if propio:
        fig.tight_layout()
        if archivo:
            fig.savefig(archivo, dpi=200)
        plt.show()
    return im

cuenca = gpd.read_file(CUENCA).to_crs(CRS)
rio = gpd.read_file(RIO).to_crs(CRS) if os.path.exists(RIO) else None

## 2. Documentación de los productos (punto 10)

In [ ]:
doc = pd.DataFrame({
    "CHIRPS v2.0": {
        "Fuente": "Climate Hazards Center, Univ. of California Santa Barbara (descarga: IRI Data Library, Columbia Univ.)",
        "Tipo": "Satélite (infrarrojo) + estaciones + climatología",
        "Resolución espacial": "0,05° (≈ 5,5 km)",
        "Resolución temporal": "diaria, pentadal, decadal y mensual (se usa mensual)",
        "Periodo disponible": "1981 – presente (50° S – 50° N, solo continente)",
        "Unidades descargadas": "mm/mes"},
    "ERA5-Land": {
        "Fuente": "ECMWF / Copernicus Climate Change Service (C3S), Climate Data Store",
        "Tipo": "Reanálisis de superficie terrestre (forzado con ERA5)",
        "Resolución espacial": "0,1° (≈ 11 km)",
        "Resolución temporal": "horaria y promedios mensuales (se usa mensual)",
        "Periodo disponible": "1950 – presente (con unos meses de rezago)",
        "Unidades descargadas": "m de agua por día (promedio mensual de acumulados diarios)"},
    "ERA5": {
        "Fuente": "ECMWF / Copernicus Climate Change Service (C3S), Climate Data Store",
        "Tipo": "Reanálisis atmosférico global",
        "Resolución espacial": "0,25° (≈ 28 km)",
        "Resolución temporal": "horaria y promedios mensuales (se usa mensual)",
        "Periodo disponible": "1940 – presente (con unos días de rezago)",
        "Unidades descargadas": "m de agua por día (promedio mensual de acumulados diarios)"},
}).T
doc["Periodo seleccionado"] = f"{PERIODO[0]} – {PERIODO[1]} (30 años, igual a las normales IDEAM)"
doc.to_csv("tabla_productos_precipitacion.csv", encoding="utf-8-sig")
doc

## 3. Descarga
Se descarga la caja que contiene la cuenca más un margen. Si un archivo ya existe, no se vuelve a descargar.
- CHIRPS (≈ 20 MB) tarda uno o dos minutos.
- ERA5 y ERA5-Land entran a una cola en el CDS. Pueden tardar desde minutos hasta horas según la demanda.

In [ ]:
import zipfile, importlib.metadata

dem, info = leer_raster(DEM)
xc, yc = coords_celdas(info)
lon_c, lat_c = a_lonlat(xc[~np.ma.getmaskarray(dem)], yc[~np.ma.getmaskarray(dem)], info["crs"])
O, E = np.floor((lon_c.min() - MARGEN_GRADOS) * 10) / 10, np.ceil((lon_c.max() + MARGEN_GRADOS) * 10) / 10
S, N = np.floor((lat_c.min() - MARGEN_GRADOS) * 10) / 10, np.ceil((lat_c.max() + MARGEN_GRADOS) * 10) / 10
print(f"Caja de descarga: lon {O} a {E}, lat {S} a {N}")

DATASET = {"ERA5L": "reanalysis-era5-land-monthly-means", "ERA5": "reanalysis-era5-single-levels-monthly-means"}
LICENCIA = {k: f"https://cds.climate.copernicus.eu/datasets/{v}?tab=download" for k, v in DATASET.items()}


def es_netcdf(ruta):
    # Un NetCDF empieza por 'CDF' (versión 3) o por la firma HDF5 (versión 4)
    if not os.path.exists(ruta) or os.path.getsize(ruta) < 1000:
        return False
    with open(ruta, "rb") as f:
        cab = f.read(8)
    return cab[:3] == b"CDF" or cab[:8] == b"\x89HDF\r\n\x1a\n"


def descomprimir_si_zip(ruta):
    # A veces el CDS entrega un .zip aunque se pida NetCDF: se extrae el .nc que trae adentro
    if os.path.exists(ruta) and zipfile.is_zipfile(ruta):
        with zipfile.ZipFile(ruta) as z:
            nc = [n for n in z.namelist() if n.endswith(".nc")]
            if not nc:
                raise ValueError(f"El zip descargado no trae archivos .nc: {z.namelist()}")
            if len(nc) > 1:
                print(f"   El zip trae {len(nc)} archivos .nc: {nc}; se usa el primero (revise si hace falta unirlos)")
            datos_nc = z.read(nc[0])
        with open(ruta, "wb") as f:
            f.write(datos_nc)
        print("   (el CDS entregó un .zip; se extrajo el NetCDF)")


def descargar_chirps(destino):
    import requests
    url = ("https://iridl.ldeo.columbia.edu/SOURCES/.UCSB/.CHIRPS/.v2p0/.monthly/.global/.precipitation/"
           f"X/{O}/{E}/RANGEEDGES/Y/{S}/{N}/RANGEEDGES/T/(Jan%20{PERIODO[0]})/(Dec%20{PERIODO[1]})/RANGEEDGES/data.nc")
    print("Descargando CHIRPS de la IRI Data Library...")
    r = requests.get(url, timeout=900)
    r.raise_for_status()
    with open(destino, "wb") as f:
        f.write(r.content)


def cliente_cds():
    # Revisa la instalación de cdsapi y la clave antes de pedir datos
    import cdsapi
    ver = importlib.metadata.version("cdsapi")
    if tuple(int(x) for x in ver.split(".")[:3] if x.isdigit()) < (0, 7, 2):
        raise RuntimeError(f"cdsapi {ver} es muy viejo para el CDS nuevo. Ejecute:  %pip install -U cdsapi  y reinicie el kernel.")
    rc = os.path.join(os.path.expanduser("~"), ".cdsapirc")
    if CDS_KEY:
        clave = str(CDS_KEY).split(":")[-1].strip()          # quita el "UID:" del formato viejo, si lo tiene
        with open(rc, "w") as f:
            f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {clave}\n")
    elif not os.path.exists(rc):
        raise RuntimeError("No hay clave del CDS. Entre a https://cds.climate.copernicus.eu, inicie sesión, abra su perfil "
                           "(https://cds.climate.copernicus.eu/profile), copie el 'Personal Access Token' y péguelo en "
                           "CDS_KEY en la celda de parámetros, entre comillas.")
    print(f"cdsapi {ver} | clave en {rc}")
    return cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=clave) if CDS_KEY else cdsapi.Client()


def explicar_error(prod, e):
    m = str(e)
    if "licen" in m.lower():
        return (f"Falta aceptar la licencia de {NOMBRE[prod]}. Abra {LICENCIA[prod]} , baje hasta 'Terms of use' y haga clic "
                "en 'Accept'. Luego vuelva a correr esta celda.")
    if "401" in m or "unauthor" in m.lower() or "not valid" in m.lower() or "authentication" in m.lower():
        return "La clave no es válida. Copie de nuevo el 'Personal Access Token' de su perfil del CDS (solo el token, sin UID)."
    if "404" in m or "not found" in m.lower():
        return "El CDS no reconoce la dirección: actualice cdsapi (%pip install -U cdsapi) y reinicie el kernel."
    if "timed out" in m.lower() or "connection" in m.lower():
        return "Problema de conexión con el CDS (red de la universidad, firewall o caída del servicio). Intente más tarde u otra red."
    return f"Error del CDS: {m[:500]}"


cliente = None
for prod, ruta in ARCHIVOS.items():
    descomprimir_si_zip(ruta)
    if es_netcdf(ruta):
        print(f"{prod}: ya existe {ruta}")
        continue
    if os.path.exists(ruta):
        os.remove(ruta)                                     # archivo incompleto o inválido de un intento anterior
    try:
        if prod == "CHIRPS":
            descargar_chirps(ruta)
        else:
            cliente = cliente or cliente_cds()
            pedido = {"product_type": ["monthly_averaged_reanalysis"], "variable": ["total_precipitation"],
                      "year": [str(a) for a in range(PERIODO[0], PERIODO[1] + 1)],
                      "month": [f"{m:02d}" for m in range(1, 13)], "time": ["00:00"],
                      "area": [float(N), float(O), float(S), float(E)],
                      "data_format": "netcdf", "download_format": "unarchived"}
            print(f"Pidiendo {NOMBRE[prod]} al CDS. Si dice 'queued' o 'running', espere: puede tardar de minutos a horas.")
            cliente.retrieve(DATASET[prod], pedido, ruta)
            descomprimir_si_zip(ruta)
        print(f"{prod}: {'descargado' if es_netcdf(ruta) else 'ATENCIÓN: el archivo no parece NetCDF'} → {ruta}")
    except Exception as e:
        print(f"{prod}: NO se pudo descargar.\n   → {explicar_error(prod, e) if prod != 'CHIRPS' else e}")


**Si alguna descarga falla:**
- **CHIRPS:** entre a la IRI Data Library → *UCSB CHIRPS v2p0 monthly global precipitation* → *Data Selection*.
  Recorte la caja que imprimió la celda anterior y el periodo, descargue en formato NetCDF y guárdelo con el nombre
  de `ARCHIVOS["CHIRPS"]`.
- **ERA5 / ERA5-Land:** en la página de cada conjunto en el CDS, use el formulario (*Download*):
  - producto *Monthly averaged reanalysis*, variable *Total precipitation*, años 1981–2010, todos los meses, hora 00:00;
  - *Sub-region extraction* con la misma caja; formato NetCDF, sin comprimir.

  Guarde los archivos con los nombres de `ARCHIVOS` y vuelva a correr desde aquí.

## 4. Lectura y conversión a mm/mes
- **CHIRPS** viene en mm/mes.
- **ERA5 y ERA5-Land**, en promedios mensuales, vienen en **metros de agua por día**: es el promedio de los
  acumulados diarios del mes. Por eso se convierten con $P_{mes}\,[mm] = tp \times 1000 \times N_{días}$.
  Este es el error más frecuente con ERA5: si no se multiplica por los días del mes, la lluvia sale unas 30 veces menor.

In [ ]:
def decodificar_tiempo(valores, unidades):
    m = re.match(r"\s*(\w+)\s+since\s+(\d{4})-(\d{1,2})-(\d{1,2})", unidades)
    if m is None:
        raise ValueError(f"Unidades de tiempo no reconocidas: {unidades}")
    paso, a, me, di = m.group(1).lower(), int(m.group(2)), int(m.group(3)), int(m.group(4))
    v = np.asarray(valores, dtype="float64")
    if paso.startswith("month"):
        k = np.floor(v + 1e-6).astype(int) + (me - 1)
        return pd.to_datetime({"year": a + k // 12, "month": k % 12 + 1, "day": 1})
    base = pd.Timestamp(a, me, di)
    unidad = {"seconds": "s", "second": "s", "hours": "h", "hour": "h", "days": "D", "day": "D", "minutes": "min"}[paso]
    return (base + pd.to_timedelta(v, unit=unidad)).to_period("M").to_timestamp()


def a_mm_mes(v):
    # Devuelve (P en mm/mes con dimensiones (tiempo, lat, lon), fechas, lat, lon)
    datos, dims = v["datos"], list(v["dims"])
    t = next(d for d in dims if d.lower() in ("time", "valid_time", "t"))
    y = next(d for d in dims if d.lower() in ("latitude", "lat", "y"))
    x = next(d for d in dims if d.lower() in ("longitude", "lon", "x"))
    for extra in [d for d in dims if d not in (t, y, x)]:          # expver, number...
        eje = dims.index(extra)
        datos = np.nanmax(datos, axis=eje) if extra == "expver" else np.take(datos, 0, axis=eje)
        dims.pop(eje)
    datos = np.transpose(datos, [dims.index(t), dims.index(y), dims.index(x)])
    fechas = pd.DatetimeIndex(decodificar_tiempo(v["coords"][t], v["unidades_coords"][t]))
    lat, lon = np.asarray(v["coords"][y], float), np.asarray(v["coords"][x], float)
    lon = np.where(lon > 180, lon - 360, lon)
    orden = np.argsort(lon)
    datos, lon = datos[:, :, orden], lon[orden]
    u = v["units"].strip().lower()
    dias = np.array([calendar.monthrange(f.year, f.month)[1] for f in fechas])[:, None, None]
    if u in ("m", "m of water equivalent", "metres", "meters"):
        datos = datos * 1000 * dias                     # ERA5: m/día -> mm/mes
    elif "mm" in u and ("day" in u or "/d" in u or "d-1" in u):
        datos = datos * dias                            # mm/día -> mm/mes
    elif "mm" not in u:
        print(f"  Aviso: unidades '{v['units']}' no reconocidas; se asumen mm/mes")
    datos = np.where(datos < 0, 0, datos)               # ERA5 puede tener valores negativos minúsculos
    dentro = (fechas.year >= PERIODO[0]) & (fechas.year <= PERIODO[1])
    return datos[dentro], fechas[dentro], lat, lon


crudo = {}
for prod in PRODUCTOS:
    if not os.path.exists(ARCHIVOS[prod]):
        print(f"{prod}: falta el archivo {ARCHIVOS[prod]}"); continue
    v = leer_variable_nc(ARCHIVOS[prod])
    crudo[prod] = a_mm_mes(v)
    p, f, la, lo = crudo[prod]
    print(f"{NOMBRE[prod]:12s} variable '{v['nombre']}' ({v['units']}) | {len(f)} meses {f.min():%Y-%m} a {f.max():%Y-%m} | "
          f"grilla {len(la)} × {len(lo)} (Δ = {abs(lo[1] - lo[0]):.3f}°)")
    if len(f) != 12 * (PERIODO[1] - PERIODO[0] + 1):
        print("   Ojo: no están todos los meses del periodo")

## 5. Climatología mensual y anual en la grilla de la cuenca
1. En la grilla original de cada producto se calcula el promedio de cada mes sobre los 30 años:
   $\overline{P}_m = \frac{1}{N}\sum_j P_{m,j}$.
2. Los 12 mapas se llevan a la grilla del DEM (~930 m):
   - con interpolación **bilineal** para los cálculos (balance, ciclos);
   - con **vecino más cercano** para los mapas, que así muestran los píxeles reales de cada producto.
3. El mapa anual es la suma de los 12 meses.

In [ ]:
mascara = np.ma.getmaskarray(dem)
anual, anual_nn, serie_cuenca = {}, {}, {}
for prod, (p, f, la, lo) in crudo.items():
    clim = np.stack([np.nanmean(p[f.month == m], axis=0) for m in range(1, 13)])      # (12, lat, lon)
    clim_dem = np.ma.masked_array(remuestrear_a_grilla(clim, lo, la, info, "bilinear"),
                                  mask=np.broadcast_to(mascara, (12,) + mascara.shape))
    guardar_raster(f"P_{prod}_mensual.tif", clim_dem, info, descripciones=MESES)
    anual[prod] = clim_dem.sum(axis=0)
    guardar_raster(f"P_{prod}_anual.tif", anual[prod], info)
    anual_nn[prod] = np.ma.masked_array(remuestrear_a_grilla(clim.sum(axis=0), lo, la, info, "nearest"), mask=mascara)
    # Serie mensual promedio de la cuenca (para ver la variabilidad interanual)
    serie = remuestrear_a_grilla(p, lo, la, info, "bilinear")
    serie_cuenca[prod] = pd.Series([np.nanmean(s[~mascara]) for s in serie], index=f)
    print(f"{NOMBRE[prod]:12s} P media anual de la cuenca: {anual[prod].mean():,.0f} mm/año")

P_ideam, _ = leer_raster(P_IDEAM)
anual["IDEAM"] = np.ma.masked_array(P_ideam, mask=mascara | np.ma.getmaskarray(P_ideam))
pd.DataFrame(serie_cuenca).to_csv("serie_mensual_cuenca_productos.csv", index_label="fecha")

## 6. Mapas de precipitación media anual multianual (punto 11)
Primero cada producto con su resolución nativa. Después los cuatro mapas (IDEAM y productos) con la **misma escala
de colores**, para comparar.

In [ ]:
from matplotlib.colors import BoundaryNorm

# Escala por clases (no lineal), la misma en todos los mapas de precipitación: detalla el rango del IDEAM y CHIRPS
# (1 000–3 500 mm/año) y a la vez deja ver los valores altos de ERA5 y ERA5-Land sin saturarlos.
orden = ["IDEAM"] + [p for p in PRODUCTOS if p in anual]
def maximo(arr):
    return float(np.nanmax(np.ma.filled(np.ma.asarray(arr, dtype=float), np.nan)))


pmax = max(maximo(anual_nn.get(p, anual[p])) for p in orden)
# La última clase termina en el máximo real dentro de la cuenca (redondeado a 1 000), no en un valor de relleno
CLASES = [0, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 5000, 6000, 8000, 10000, 15000, 20000]
tope = float(np.ceil(pmax / 1000) * 1000)
CLASES = [c for c in CLASES if c < tope] + [tope]
NORMA = BoundaryNorm(CLASES, plt.get_cmap("YlGnBu").N)


def barra(fig, im, ax):
    return fig.colorbar(im, ax=ax, shrink=0.6, ticks=CLASES, spacing="uniform", label="Precipitación media anual (mm/año)")


# Cada producto en su resolución nativa
for prod in anual_nn:
    fig, ax = plt.subplots(figsize=(6, 9))
    im = mapa(anual_nn[prod], info, f"Precipitación media anual {PERIODO[0]}–{PERIODO[1]}\n{NOMBRE[prod]} (resolución nativa) · "
              f"máx. {maximo(anual_nn[prod]):,.0f} mm/año", "", cuenca, rio, cmap="YlGnBu", ax=ax)
    im.set_norm(NORMA)
    barra(fig, im, ax)
    fig.tight_layout(); fig.savefig(f"fig_mapa_P_{prod}.png", dpi=200)
    plt.show()

# Los cuatro mapas juntos, con la misma escala
fig, axs = plt.subplots(1, len(orden), figsize=(4.2 * len(orden), 8))
for ax, prod in zip(np.atleast_1d(axs), orden):
    arr = anual[prod] if prod == "IDEAM" else anual_nn[prod]
    im = mapa(arr, info, f"{NOMBRE[prod]}\nmedia {anual[prod].mean():,.0f} mm/año\nmáx. {maximo(arr):,.0f} mm/año", "", cuenca, rio,
              cmap="YlGnBu", ax=ax)
    im.set_norm(NORMA)
for ax in np.atleast_1d(axs)[1:]:
    ax.set_ylabel(""); ax.tick_params(labelleft=False)
barra(fig, im, axs)
fig.savefig("fig_mapas_P_productos.png", dpi=200, bbox_inches="tight")
plt.show()

### Verificación de la lectura de los productos
Comprueba, con números, que cada archivo se leyó en la ubicación correcta. Para eso se busca el máximo de cada
producto en toda la caja descargada, no solo en la cuenca. La zona más lluviosa de Colombia es la costa Pacífica, así
que en **CHIRPS** el máximo debe quedar hacia el oeste (longitud cercana a −77°). Si en **ERA5 y ERA5-Land** también
queda allí, los archivos están bien leídos y los valores altos dentro de la cuenca son del propio producto, no un error
del procesamiento.

In [ ]:
# Verificación (solo números): ¿cada archivo se leyó en la ubicación correcta?
# Se mira cada producto en TODA la caja descargada: su máximo debe quedar hacia la costa Pacífica (oeste, longitudes
# cercanas a −77°), que es la zona más lluviosa de Colombia, como en CHIRPS.
for prod, (p, f, la, lo) in crudo.items():
    clim = np.stack([np.nanmean(p[f.month == m], axis=0) for m in range(1, 13)]).sum(axis=0)
    k = np.unravel_index(np.nanargmax(clim), clim.shape)
    print(f"{NOMBRE[prod]:12s} lat {la[0]:.2f} → {la[-1]:.2f} | lon {lo[0]:.2f} → {lo[-1]:.2f} | "
          f"máximo de la caja: {np.nanmax(clim):,.0f} mm/año en ({la[k[0]]:.2f}° N, {lo[k[1]]:.2f}°)")

In [ ]:
area_celda = abs(info["transform"].a * info["transform"].e) / 1e6
area_cuenca = (~mascara).sum() * area_celda
res = {"CHIRPS": 0.05, "ERA5L": 0.1, "ERA5": 0.25}
resumen = pd.DataFrame({NOMBRE[p]: {
    "P media anual cuenca (mm/año)": anual[p].mean(),
    "diferencia vs IDEAM (%)": 100 * (anual[p].mean() / anual["IDEAM"].mean() - 1),
    "P mínima (mm/año)": anual[p].min(), "P máxima (mm/año)": anual[p].max(),
    "píxeles nativos que cubren la cuenca (aprox.)": (area_cuenca / (res[p] * 111.2) ** 2) if p in res else np.nan,
} for p in orden}).T
display(resumen.round(1))

if serie_cuenca:
    s = pd.DataFrame(serie_cuenca)
    fig, ax = plt.subplots(figsize=(10, 4))
    s.groupby(s.index.year).sum().rename(columns=NOMBRE).plot(ax=ax, marker="o", ms=3)
    ax.axhline(anual["IDEAM"].mean(), color="k", ls="--", lw=1, label="IDEAM (normal 1981–2010)")
    ax.set(xlabel="Año", ylabel="P anual de la cuenca (mm)", title="Precipitación anual de la cuenca por producto")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig("fig_serie_anual_productos.png", dpi=200)
    plt.show()

**Para el informe (punto 11):** compare los patrones espaciales de los mapas.
- ¿Todos muestran el valle seco y el norte muy lluvioso?
- ¿Algún producto suaviza demasiado la lluvia por su resolución? ERA5 cubre la cuenca con pocos píxeles de ~28 km.
- ¿Alguno sobrestima la lluvia sobre las cordilleras? Es frecuente en los reanálisis sobre los Andes.

La serie anual muestra además los años secos y húmedos asociados a El Niño y La Niña, por ejemplo 1997–98 y 2010.